In [ ]:
import time
from urllib.parse import urljoin

from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

NOMBRE_GRUPO = "energia-y-sustentabilidad-1"
CATEGORIA_ACTUAL = "Travel"
CATEGORIA_URL = "https://books.toscrape.com/catalogue/category/books/travel_2/index.html"
MONGO_URI = "mongodb://database:27017/"

client = MongoClient(MONGO_URI)
db = client["proyecto_bigdata"]
coleccion = db["datos_libros"]

print("Conexion exitosa con MongoDB")
print(f"Grupo: {NOMBRE_GRUPO} | Categoria: {CATEGORIA_ACTUAL}")

In [ ]:
options = Options()
options.binary_location = "/usr/bin/brave-browser"
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1366,768")

driver = webdriver.Chrome(options=options)
registros = []
current_url = CATEGORIA_URL
paginas_visitadas = 0

try:
    while current_url:
        driver.get(current_url)
        paginas_visitadas += 1

        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".product_pod"))
        )

        libros = driver.find_elements(By.CSS_SELECTOR, ".product_pod")
        fecha_captura = time.strftime("%Y-%m-%d %H:%M:%S")

        for libro in libros:
            titulo = libro.find_element(By.CSS_SELECTOR, "h3 a").get_attribute("title").strip()
            precio_texto = libro.find_element(By.CSS_SELECTOR, ".price_color").text
            valor = float(precio_texto.replace("Â£", "").replace("£", "").strip())

            registros.append({
                "item": titulo,
                "valor": valor,
                "categoria": CATEGORIA_ACTUAL,
                "grupo": NOMBRE_GRUPO,
                "fecha_captura": fecha_captura
            })

        siguiente = driver.find_elements(By.CSS_SELECTOR, ".next a")
        current_url = urljoin(current_url, siguiente[0].get_attribute("href")) if siguiente else None
finally:
    driver.quit()

coleccion.delete_many({"grupo": NOMBRE_GRUPO, "categoria": CATEGORIA_ACTUAL})

if registros:
    coleccion.insert_many(registros)
    print(f"Proceso finalizado: {len(registros)} libros insertados desde {paginas_visitadas} pagina(s).")
    print("Primer registro:")
    print(registros[0])
else:
    print("No se encontraron libros para insertar.")